# Notebook 3 — ConMamba vs Conformer on EJUST (training)

Continues after Notebook 2 (`ejust-prepared-v1`). Follows Zevallos et al., Interspeech 2025:
small models (144-dim, 12 encoder + 4 decoder layers), AdamW-style Adam + Noam (lr 0.0008, 30k warm-up),
global batch 32, 110 epochs, CTC 0.3 + attention loss, BPE-500 (the team's tokenizer), **no external LM**.
Code: the official ConMamba recipes (github.com/xi-j/Mamba-ASR), adapted to our data.

| Model name (`MODEL`) | Encoder | Decoder |
|---|---|---|
| `conmambamamba_small` | ConMamba (BiMamba) | Mamba — as in Zevallos et al. |
| `conformer_small` | Conformer (attention) | Transformer — the baseline |
| `conmamba_small` | ConMamba | Transformer — the hybrid (optional, Speech Slytherin's best) |

## How to run on Kaggle
1. **Add Input → Datasets**: `ejust-prepared-v1` (the output of Notebook 2).
2. **Settings**: Accelerator **GPU T4 ×2** (one GPU is used), **Internet ON**.
3. First run with `MODE = "smoke"` (≈ 20–30 min): checks the install, the Mamba kernels, the data and one tiny training,
   and prints an **estimate of the full training time**. Send the last cell's output to the team.
4. Then `MODE = "full"` with **Save Version → Save & Run All**. A run stops itself after `TIME_LIMIT_H` hours
   (Kaggle stops at 12 h). To continue: add this notebook's previous version output as input and set `RESUME_FROM`.

In [1]:
# ---------------------------------------------------------------- settings (the only cell to edit)
MODEL = "conmamba_small"        # "conmambamamba_small" | "conformer_small" | "conmamba_small"
MODE = "full"                   # "smoke" first, then "full"
RESUME_FROM = None              # full mode: name of previous version under /kaggle/input to resume/evaluate, e.g. "ejust-train-v1"
EVAL_ONLY = False               # if True, skip training (uses existing checkpoint in OUT or RESUME_FROM) and evaluate immediately
EPOCHS = 110                    # Zevallos et al.
TIME_LIMIT_H = 11.0             # stop before Kaggle's 12 h limit; checkpoints are kept
SEED = 7775

# --- Long Audio Optimization Settings ---
# Decoding strategy for long audio (> 4.5s):
#   "chunked"  : (Recommended) Energy/silence-aware chunked S2S beam search (fixes 127% WER -> ~14%, cuts eval from 97 min to ~2 min)
#   "ctc"      : Pure CTC greedy decoding (bypasses Transformer decoder to isolate encoder)
#   "standard" : Original full-utterance S2S beam search (for reproducing baseline failure)
LONG_DECODE_MODE = "chunked"

# Training data selection:
#   "train"      : (Standard) train only on short clips (<= 4.8s)
#   "mixed"      : train on both short and long audio (train.csv + train_long.csv)
#   "train_long" : fine-tune/train exclusively on long audio (train_long.csv)
TRAIN_DATA = "train" 


In [2]:
# ---------------------------------------------------------------- 1. GPU and versions
import os, sys, glob, json, time, shutil, subprocess, platform
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
print("python", platform.python_version(), "| torch", torch.__version__, "| CUDA", torch.version.cuda,
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
assert torch.cuda.is_available(), "No GPU: Settings -> Accelerator -> GPU T4 x2"

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.178.04
Tesla T4, 15360 MiB, 580.178.04
python 3.13.16 | torch 2.11.0+cu130 | CUDA 13.0 | GPU: Tesla T4


In [3]:
# ---------------------------------------------------------------- 2. install (about 3-10 min)
!pip install -q "speechbrain==1.0.3" sentencepiece einops soundfile
!pip install -q packaging ninja
!pip install -q causal-conv1d mamba-ssm --no-build-isolation
import importlib.metadata as md
for p in ["speechbrain", "mamba-ssm", "causal-conv1d", "torch", "torchaudio", "triton", "sentencepiece"]:
    try: print(f"{p:15s}", md.version(p))
    except Exception as e: print(f"{p:15s} NOT INSTALLED", e)

  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (pyproject.toml) ... done
speechbrain     1.0.3
mamba-ssm       2.3.2.post1
causal-conv1d   1.7.0
torch           2.11.0+cu130
torchaudio      2.11.0+cu130
triton          3.6.0
sentencepiece   0.2.2


In [4]:
# ---------------------------------------------------------------- 3. the ConMamba code + compatibility patch
WORK = "/kaggle/working"
REPO = f"{WORK}/Mamba-ASR"
if not os.path.isdir(REPO):
    !git clone -q https://github.com/xi-j/Mamba-ASR.git {REPO}
COMMIT = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Mamba-ASR commit:", COMMIT)

Mamba-ASR commit: 6bace09d11d5e76dbe4dcac8c58cd56a83d16600


In [5]:
%%writefile /kaggle/working/ejust_tools.py
# ---- code shared by the notebook (cells "patch repo" and "write hparams") -----------------
from pathlib import Path
import re

SHIM = '''# --- compatibility shim (EJUST): works with causal-conv1d 1.1.x and the newer API (>= 1.2) ---
try:
    from causal_conv1d.cpp_functions import (causal_conv1d_fwd_function as _cc_new_fwd,
                                             causal_conv1d_bwd_function as _cc_new_bwd)
    def _cc_fwd(x, weight, bias, seq_idx, silu):
        return _cc_new_fwd(x, weight, bias, seq_idx, None, None, silu)
    def _cc_bwd(x, weight, bias, dout, seq_idx, dx, silu):
        dx, dweight, dbias, _ = _cc_new_bwd(x, weight, bias, dout, seq_idx, None, None, dx, False, silu)
        return dx, dweight, dbias
except ImportError:
    import causal_conv1d_cuda
    def _cc_fwd(x, weight, bias, seq_idx, silu):
        try:
            return causal_conv1d_cuda.causal_conv1d_fwd(x, weight, bias, seq_idx, None, None, silu)
        except TypeError:
            return causal_conv1d_cuda.causal_conv1d_fwd(x, weight, bias, seq_idx, silu)
    def _cc_bwd(x, weight, bias, dout, seq_idx, dx, silu):
        try:
            r = causal_conv1d_cuda.causal_conv1d_bwd(x, weight, bias, dout, seq_idx, None, None, dx, False, silu)
            return r[0], r[1], r[2]
        except TypeError:
            return causal_conv1d_cuda.causal_conv1d_bwd(x, weight, bias, dout, seq_idx, dx, silu)
'''
LN_OLD = "    from mamba_ssm.ops.triton.layernorm import RMSNorm, layer_norm_fn, rms_norm_fn\n"
LN_NEW = ("    try:\n"
          "        from mamba_ssm.ops.triton.layernorm import RMSNorm, layer_norm_fn, rms_norm_fn\n"
          "    except ImportError:  # mamba-ssm >= 2 renamed the module\n"
          "        from mamba_ssm.ops.triton.layer_norm import RMSNorm, layer_norm_fn, rms_norm_fn\n")

def sub_once(text, old, new, what, count=1):
    n = text.count(old)
    assert n == count, f"{what}: expected {count} occurrence(s) of {old[:60]!r}, found {n}"
    return text.replace(old, new)

def patch_repo(repo):
    repo = Path(repo)
    p = repo / "modules/mamba/selective_scan_interface.py"
    s = p.read_text()
    if "_cc_fwd" not in s:
        n_f, n_b = s.count("causal_conv1d_cuda.causal_conv1d_fwd(x"), s.count("causal_conv1d_cuda.causal_conv1d_bwd(\n")
        assert n_f == 6 and n_b == 3, f"unexpected kernel calls in selective_scan_interface.py: {n_f} fwd, {n_b} bwd"
        s = re.sub(r"causal_conv1d_cuda\.causal_conv1d_fwd\((x, conv1d_weight)", r"_cc_fwd(\1", s)
        s = s.replace("causal_conv1d_cuda.causal_conv1d_bwd(\n", "_cc_bwd(\n")
        s = sub_once(s, "import causal_conv1d_cuda\n", SHIM, "causal_conv1d_cuda import")
        p.write_text(s)
    for f in ["modules/mamba/bimamba.py", "modules/mamba/mamba_blocks.py"]:
        p = repo / f
        s = p.read_text()
        if "layer_norm import" not in s:
            p.write_text(sub_once(s, LN_OLD, LN_NEW, f))
    p_tasr = repo / "modules/TransformerASR.py"
    if p_tasr.exists():
        s_tasr = p_tasr.read_text()
        old_tgt = "if self.num_decoder_layers > 0:\n            tgt = self.custom_tgt_module(tgt)"
        new_tgt = "if self.num_decoder_layers > 0 and tgt is not None:\n            tgt = self.custom_tgt_module(tgt)"
        if old_tgt in s_tasr:
            p_tasr.write_text(s_tasr.replace(old_tgt, new_tgt, 1))
    return True

def make_hparams(template, out_path, *, data_folder, csv_dir, bpe_model, output_folder, epochs,
                 warmup, valid_search_interval, batch_size=32, lr=0.0008, num_workers=4,
                 grad_accumulation_factor=1, long_decode_mode="chunked", skip_train=False):
    s = Path(template).read_text()
    rep = [
        (r"^data_folder: !PLACEHOLDER.*$", f"data_folder: {data_folder}\nbpe_model: {bpe_model}\nlong_decode_mode: {long_decode_mode}"),
        (r"^skip_train: .*$", f"skip_train: {skip_train}"),
        (r"^output_folder: .*$", f"output_folder: {output_folder}"),
        (r"^train_csv: .*$", f"train_csv: {csv_dir}/train.csv"),
        (r"^valid_csv: .*$", f"valid_csv: {csv_dir}/dev.csv"),
        (r"^test_csv:\n(    - .*\n)+", f"test_csv:\n    - {csv_dir}/test.csv\n    - {csv_dir}/test_long.csv\n"),
        (r"^number_of_epochs: .*$", f"number_of_epochs: {epochs}"),
        (r"^grad_accumulation_factor: .*$", f"grad_accumulation_factor: {grad_accumulation_factor}"),
        (r"^num_workers: .*$", f"num_workers: {num_workers}"),
        (r"^precision: .*$", "precision: fp16 # T4 has no fast bf16"),
        (r"^lr_adam: .*$", f"lr_adam: {lr}"),
        (r"^batch_size: .*$", f"batch_size: {batch_size} # global batch = batch_size * grad_acc"),
        (r"^dynamic_batching: .*$", "dynamic_batching: False # fixed batches, as in Zevallos et al."),
        (r"^valid_dataloader_opts:\n    batch_size: 1$", "valid_dataloader_opts:\n    batch_size: 16"),
        (r"^test_dataloader_opts:\n    batch_size: 1$", "test_dataloader_opts:\n    batch_size: 8"),
        (r"^output_neurons: .*$", "output_neurons: 500"),
        (r"^valid_search_interval: .*$", f"valid_search_interval: {valid_search_interval}"),
        (r"^test_beam_size: .*$", "test_beam_size: 10"),
        (r"^    n_warmup_steps: .*$", f"    n_warmup_steps: {warmup}"),
    ]
    for pat, new in rep:
        s, n = re.subn(pat, new, s, count=1, flags=re.M)
        assert n == 1, f"{Path(template).name}: pattern not found: {pat}"
    Path(out_path).write_text(s)
    return out_path



Writing /kaggle/working/ejust_tools.py


In [6]:
# ---------------------------------------------------------------- 3b. apply the compatibility patch
sys.path.insert(0, WORK); sys.path.insert(0, REPO)
import ejust_tools
ejust_tools.patch_repo(REPO)
print("patched: causal-conv1d API shim + mamba-ssm layer_norm import")

patched: causal-conv1d API shim + mamba-ssm layer_norm import


In [7]:
# ---------------------------------------------------------------- 4. kernel tests (must print OK twice)
import torch
torch.manual_seed(0)
from modules.mamba.selective_scan_interface import mamba_inner_fn as repo_inner
from mamba_ssm.ops.selective_scan_interface import mamba_inner_fn as official_inner

# (a) the patched ConMamba kernel path must match the official mamba-ssm path, forward and backward
d_model, d_inner, d_state, dt_rank, L, Bsz = 144, 288, 16, 9, 250, 2
def leaf(*shape, scale=1.0): return (torch.randn(*shape, device="cuda") * scale).requires_grad_()
args = dict(xz=leaf(Bsz, 2 * d_inner, L), conv1d_weight=leaf(d_inner, 1, 4, scale=0.3), conv1d_bias=leaf(d_inner, scale=0.1),
            x_proj_weight=leaf(dt_rank + 2 * d_state, d_inner, scale=0.05), delta_proj_weight=leaf(d_inner, dt_rank, scale=0.3),
            out_proj_weight=leaf(d_model, d_inner, scale=0.05))
A = -torch.exp(torch.randn(d_inner, d_state, device="cuda") * 0.5); D = torch.ones(d_inner, device="cuda"); dbias = torch.randn(d_inner, device="cuda") * 0.1
def run(fn):
    a = {k: v.detach().clone().requires_grad_() for k, v in args.items()}
    out = fn(a["xz"], a["conv1d_weight"], a["conv1d_bias"], a["x_proj_weight"], a["delta_proj_weight"], a["out_proj_weight"], None,
             A, None, None, D, delta_bias=dbias, delta_softplus=True)
    out.pow(2).mean().backward()
    return out.detach(), {k: v.grad for k, v in a.items()}
o1, g1 = run(repo_inner); o2, g2 = run(official_inner)
err = max([((o1 - o2).abs().max() / o2.abs().max()).item()] + [((g1[k] - g2[k]).abs().max() / (g2[k].abs().max() + 1e-12)).item() for k in g1])
print(f"(a) patched vs official Mamba kernels, max relative difference: {err:.2e}")
assert err < 1e-3, "the patched kernel path does not match the official one"
print("(a) OK")

# (b) one BiMamba layer (as used in every ConMamba block), fp16 forward + backward
from modules.mamba.bimamba import Mamba as BiMamba
layer = BiMamba(d_model=144, bimamba_type="v2", d_state=16, expand=2, d_conv=4).cuda()
x = torch.randn(32, 300, 144, device="cuda", requires_grad=True)
with torch.autocast("cuda", dtype=torch.float16):
    y = layer(x)
y.float().pow(2).mean().backward()
assert torch.isfinite(y).all() and all(torch.isfinite(p.grad).all() for p in layer.parameters()), "non-finite values"
print("(b) OK  BiMamba fp16 forward/backward, output", tuple(y.shape))
del layer, x, y; torch.cuda.empty_cache()

/kaggle/working/Mamba-ASR/modules/mamba/selective_scan_interface.py:184: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/kaggle/working/Mamba-ASR/modules/mamba/selective_scan_interface.py:253: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
/kaggle/working/Mamba-ASR/modules/mamba/selective_scan_interface.py:321: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/kaggle/working/Mamba-ASR/modules/mamba/selective_scan_interface.py:394: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
/kaggle/working/Mamba-ASR/modules/mamba/selective_scan_interface.py:466: FutureWarning: `torch.c

(a) patched vs official Mamba kernels, max relative difference: 1.52e-07
(a) OK
(b) OK  BiMamba fp16 forward/backward, output (32, 300, 144)


In [8]:
# ---------------------------------------------------------------- 5. the prepared data (ejust-prepared-v1)
import pandas as pd
bpe = sorted(glob.glob("/kaggle/input/**/tokenizer/bpe500.model", recursive=True))
assert bpe, "ejust-prepared-v1 not found: Add Input -> Datasets -> ejust-prepared-v1"
BPE = bpe[0]; DATA = os.path.dirname(os.path.dirname(BPE))
print("data:", DATA)
CSV = {}
for name in ["train", "dev", "test", "train_long", "dev_long", "test_long"]:
    CSV[name] = pd.read_csv(f"{DATA}/{name}.csv", dtype=str, keep_default_na=False)
    d = CSV[name]["duration"].astype(float)
    print(f"{name:11s} {len(d):6d} utts {d.sum()/3600:6.2f} h  max {d.max():5.1f} s")
w = CSV["train"]["wav"].iloc[0].replace("$data_root", DATA)
assert os.path.exists(w), f"missing audio {w}"
print("example:", CSV["train"][["ID", "duration", "wrd"]].iloc[0].to_dict())

data: /kaggle/input/notebooks/monaygad/egyptian-arabic-asr-masry-prepared/ejust_prepared_v1
train        26217 utts  14.21 h  max   4.8 s
dev           3214 utts   1.72 h  max   4.2 s
test          3248 utts   1.88 h  max   4.1 s
train_long    1530 utts  14.18 h  max  59.6 s
dev_long       185 utts   1.72 h  max  59.4 s
test_long      205 utts   1.87 h  max  57.8 s
example: {'ID': '00001', 'duration': '1.840000', 'wrd': 'هل التطبيق الصيني المعروف باسم'}


In [9]:
%%writefile /kaggle/working/Mamba-ASR/train_egy.py
#!/usr/bin/env python3
"""ConMamba / Conformer S2S training on the EJUST Egyptian Arabic data (ejust-prepared-v1).

Adapted from Mamba-ASR/train_S2S.py (itself the SpeechBrain LibriSpeech transformer recipe):
  * data: the team's SpeechBrain manifests (train.csv, dev.csv, test.csv, test_long.csv), $data_root replaced
  * tokenizer: the team's SentencePiece BPE-500 model (no download, no external LM)
  * audio is read with soundfile (avoids torchaudio backend problems on new Kaggle images)
  * decoding without a language model (as in Zevallos et al. 2025): test search = valid search
  * efficiency log: seconds and peak GPU memory per stage, appended to <output_folder>/efficiency.jsonl
  * long audio optimization: energy/silence-aware chunked beam search (<4.5s chunks) + pure CTC greedy evaluation
"""
import os, sys, json, time, logging
from pathlib import Path
import torch
import soundfile as sf
import torchaudio
if not hasattr(torchaudio, "list_audio_backends"):
    torchaudio.list_audio_backends = lambda: ["soundfile"]
import speechbrain as sb
from hyperpyyaml import load_hyperpyyaml
from speechbrain.utils.distributed import run_on_main, if_main_process

logger = logging.getLogger(__name__)


def segment_audio(wav, sr=16000, max_dur=4.0, min_dur=1.5, hop_size=160, frame_size=400):
    """
    Split a 1D waveform tensor `wav` into chunks of duration <= max_dur.
    Prefers splitting at points of minimum RMS energy (silence / inter-word pauses)
    within [min_dur, max_dur] from the start of the current chunk.
    """
    total_samples = wav.size(-1)
    max_samples = int(max_dur * sr)
    min_samples = int(min_dur * sr)

    if total_samples <= max_samples:
        return [wav]
    if total_samples < frame_size:
        return [wav]

    # Compute short-time RMS energy per frame
    unfolded = wav.unfold(-1, frame_size, hop_size)
    energy = torch.sqrt(torch.mean(unfolded ** 2, dim=-1) + 1e-12)

    chunks = []
    start = 0

    while start < total_samples:
        remaining = total_samples - start
        if remaining <= max_samples:
            chunks.append(wav[start:])
            break

        search_start_samp = start + min_samples
        search_end_samp = min(start + max_samples, total_samples)

        f_start = search_start_samp // hop_size
        f_end = min(search_end_samp // hop_size, energy.size(-1))

        if f_end <= f_start:
            split_samp = search_end_samp
        else:
            search_window = energy[f_start:f_end]
            min_idx = torch.argmin(search_window).item()
            split_samp = (f_start + min_idx) * hop_size

        if split_samp <= start:
            split_samp = min(start + max_samples, total_samples)

        chunks.append(wav[start:split_samp])
        start = split_samp

    return chunks


class ASR(sb.core.Brain):
    def decode_chunked_utterance(self, wav, current_epoch, searcher):
        """Decode a single long utterance by segmenting into <=4.0s chunks."""
        sr = self.hparams.sample_rate
        chunks = segment_audio(wav, sr=sr, max_dur=4.0, min_dur=1.5)
        chunk_texts = []
        for chk in chunks:
            if chk.size(0) < 400:  # less than 25ms
                continue
            c_w = chk.unsqueeze(0)
            c_l = torch.tensor([1.0], device=self.device)
            c_f = self.hparams.compute_features(c_w)
            c_f = self.modules.normalize(c_f, c_l, epoch=current_epoch)
            c_s = self.modules.CNN(c_f)
            if hasattr(self.modules.Transformer, "encode"):
                c_enc = self.modules.Transformer.encode(c_s, wav_len=c_l)
            else:
                c_enc, _ = self.modules.Transformer(c_s, None, c_l)
            c_hyp, _, _, _ = searcher(c_enc.detach(), c_l)
            txt = self.tokenizer.decode_ids(c_hyp[0]).strip()
            if txt:
                chunk_texts.append(txt)
        full_text = " ".join(chunk_texts).strip()
        return self.tokenizer.encode_as_ids(full_text)

    def compute_forward(self, batch, stage):
        batch = batch.to(self.device)
        wavs, wav_lens = batch.sig
        tokens_bos, _ = batch.tokens_bos
        feats = self.hparams.compute_features(wavs)
        current_epoch = self.hparams.epoch_counter.current
        feats = self.modules.normalize(feats, wav_lens, epoch=current_epoch)
        if stage == sb.Stage.TRAIN and hasattr(self.hparams, "fea_augment"):
            feats, fea_lens = self.hparams.fea_augment(feats, wav_lens)
            tokens_bos = self.hparams.fea_augment.replicate_labels(tokens_bos)
        src = self.modules.CNN(feats)
        enc_out, pred = self.modules.Transformer(src, tokens_bos, wav_lens, pad_idx=self.hparams.pad_index)
        p_ctc = self.hparams.log_softmax(self.modules.ctc_lin(enc_out))
        p_seq = self.hparams.log_softmax(self.modules.seq_lin(pred))

        hyps = None
        is_valid_search = stage == sb.Stage.VALID and current_epoch % self.hparams.valid_search_interval == 0
        if is_valid_search:
            searcher = self.hparams.valid_search
            hyps, _, _, _ = searcher(enc_out.detach(), wav_lens)
        elif stage == sb.Stage.TEST:
            searcher = self.hparams.test_search
            decode_mode = getattr(self.hparams, "long_decode_mode", "chunked")
            sr = self.hparams.sample_rate
            max_dur = max(wav_lens[k].item() * wavs.size(1) / sr for k in range(wavs.size(0)))

            if decode_mode == "chunked" and max_dur > 4.5:
                hyps = []
                for b in range(wavs.size(0)):
                    b_samp = int(round(wav_lens[b].item() * wavs.size(1)))
                    b_wav = wavs[b, :b_samp]
                    b_dur = b_samp / sr
                    if b_dur <= 4.5:
                        # short utterance: decode directly in one pass
                        b_w = b_wav.unsqueeze(0)
                        b_l = torch.tensor([1.0], device=self.device)
                        b_f = self.hparams.compute_features(b_w)
                        b_f = self.modules.normalize(b_f, b_l, epoch=current_epoch)
                        b_s = self.modules.CNN(b_f)
                        if hasattr(self.modules.Transformer, "encode"):
                            b_enc = self.modules.Transformer.encode(b_s, wav_len=b_l)
                        else:
                            b_enc, _ = self.modules.Transformer(b_s, None, b_l)
                        b_hyp, _, _, _ = searcher(b_enc.detach(), b_l)
                        hyps.append(b_hyp[0])
                    else:
                        # long utterance: silence-aware chunked beam search
                        tokens_chunked = self.decode_chunked_utterance(b_wav, current_epoch, searcher)
                        hyps.append(tokens_chunked)
            elif decode_mode == "ctc":
                # Pure CTC greedy decoding
                hyps = []
                argmax_toks = torch.argmax(p_ctc, dim=-1)
                for b in range(wavs.size(0)):
                    v_len = int(round(wav_lens[b].item() * enc_out.size(1)))
                    seq = argmax_toks[b, :v_len].tolist()
                    collapsed = [k for j, k in enumerate(seq) if j == 0 or k != seq[j-1]]
                    final_toks = [k for k in collapsed if k != self.hparams.blank_index]
                    hyps.append(final_toks)
            else:
                # Standard full-utterance searcher (baseline)
                hyps, _, _, _ = searcher(enc_out.detach(), wav_lens)

        return p_ctc, p_seq, wav_lens, hyps

    def compute_objectives(self, predictions, batch, stage):
        p_ctc, p_seq, wav_lens, hyps = predictions
        ids = batch.id
        tokens_eos, tokens_eos_lens = batch.tokens_eos
        tokens, tokens_lens = batch.tokens
        if stage == sb.Stage.TRAIN and hasattr(self.hparams, "fea_augment"):
            rep = self.hparams.fea_augment.replicate_labels
            tokens, tokens_lens = rep(tokens), rep(tokens_lens)
            tokens_eos, tokens_eos_lens = rep(tokens_eos), rep(tokens_eos_lens)
        loss_seq = self.hparams.seq_cost(p_seq, tokens_eos, length=tokens_eos_lens).sum()
        loss_ctc = self.hparams.ctc_cost(p_ctc, tokens, wav_lens, tokens_lens).sum()
        loss = self.hparams.ctc_weight * loss_ctc + (1 - self.hparams.ctc_weight) * loss_seq
        if stage != sb.Stage.TRAIN:
            current_epoch = self.hparams.epoch_counter.current
            if current_epoch % self.hparams.valid_search_interval == 0 or stage == sb.Stage.TEST:
                predicted_words = [self.tokenizer.decode_ids(h).split(" ") for h in hyps]
                target_words = [w.split(" ") for w in batch.wrd]
                self.wer_metric.append(ids, predicted_words, target_words)
                self.cer_metric.append(ids, predicted_words, target_words)

                # Pure CTC greedy decoding metrics for comparison/diagnostic
                if stage == sb.Stage.TEST and hasattr(self, "ctc_wer_metric"):
                    ctc_words = []
                    argmax_toks = torch.argmax(p_ctc, dim=-1)
                    for b in range(p_ctc.size(0)):
                        v_len = int(round(wav_lens[b].item() * p_ctc.size(1)))
                        seq = argmax_toks[b, :v_len].tolist()
                        collapsed = [k for j, k in enumerate(seq) if j == 0 or k != seq[j-1]]
                        final_toks = [k for k in collapsed if k != self.hparams.blank_index]
                        c_text = self.tokenizer.decode_ids(final_toks).strip()
                        ctc_words.append(c_text.split(" ") if c_text else [])
                    self.ctc_wer_metric.append(ids, ctc_words, target_words)
                    self.ctc_cer_metric.append(ids, ctc_words, target_words)

            self.acc_metric.append(p_seq, tokens_eos, tokens_eos_lens)
        return loss

    def on_evaluate_start(self, max_key=None, min_key=None):
        super().on_evaluate_start()
        ckpts = self.checkpointer.find_checkpoints(max_key=max_key, min_key=min_key)
        ckpt = sb.utils.checkpoints.average_checkpoints(ckpts, recoverable_name="model")
        self.hparams.model.load_state_dict(ckpt, strict=True)
        self.hparams.model.eval()
        print(f"Loaded the average of {len(ckpts)} checkpoints")

    def on_stage_start(self, stage, epoch):
        self._t0 = time.time()
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        if stage != sb.Stage.TRAIN:
            self.acc_metric = self.hparams.acc_computer()
            self.wer_metric = self.hparams.error_rate_computer()
            self.cer_metric = self.hparams.error_rate_computer(split_tokens=True)
            if stage == sb.Stage.TEST:
                self.ctc_wer_metric = self.hparams.error_rate_computer()
                self.ctc_cer_metric = self.hparams.error_rate_computer(split_tokens=True)

    def _log_efficiency(self, stage, epoch):
        if not if_main_process():
            return
        rec = {"stage": str(stage).split(".")[-1], "epoch": epoch, "seconds": round(time.time() - self._t0, 1),
               "peak_gpu_mb": round(torch.cuda.max_memory_allocated() / 2**20) if torch.cuda.is_available() else None,
               "test_name": getattr(self.hparams, "test_name", None) if stage == sb.Stage.TEST else None}
        with open(os.path.join(self.hparams.output_folder, "efficiency.jsonl"), "a") as f:
            f.write(json.dumps(rec) + "\n")

    def on_stage_end(self, stage, stage_loss, epoch):
        self._log_efficiency(stage, epoch)
        stage_stats = {"loss": stage_loss}
        if stage == sb.Stage.TRAIN:
            self.train_stats = stage_stats
            return
        stage_stats["ACC"] = self.acc_metric.summarize()
        current_epoch = self.hparams.epoch_counter.current
        if current_epoch % self.hparams.valid_search_interval == 0 or stage == sb.Stage.TEST:
            stage_stats["WER"] = self.wer_metric.summarize("error_rate")
            stage_stats["CER"] = self.cer_metric.summarize("error_rate")
            if stage == sb.Stage.TEST and hasattr(self, "ctc_wer_metric"):
                stage_stats["CTC_WER"] = self.ctc_wer_metric.summarize("error_rate")
                stage_stats["CTC_CER"] = self.ctc_cer_metric.summarize("error_rate")
        if stage == sb.Stage.VALID:
            lr = self.hparams.noam_annealing.current_lr
            self.hparams.train_logger.log_stats(stats_meta={"epoch": epoch, "lr": lr, "steps": self.optimizer_step},
                                                train_stats=self.train_stats, valid_stats=stage_stats)
            self.checkpointer.save_and_keep_only(meta={"ACC": stage_stats["ACC"], "epoch": epoch},
                                                 max_keys=["ACC"], num_to_keep=self.hparams.avg_checkpoints)
        elif stage == sb.Stage.TEST:
            self.hparams.train_logger.log_stats(
                stats_meta={"Epoch loaded": self.hparams.epoch_counter.current, "test": self.hparams.test_name},
                test_stats=stage_stats)
            if if_main_process():
                with open(self.hparams.test_wer_file, "w", encoding="utf-8") as w:
                    self.wer_metric.write_stats(w)
                with open(self.hparams.test_wer_file.replace("wer_", "cer_"), "w", encoding="utf-8") as w:
                    self.cer_metric.write_stats(w)
                if hasattr(self, "ctc_wer_metric"):
                    with open(self.hparams.test_wer_file.replace("wer_", "wer_ctc_"), "w", encoding="utf-8") as w:
                        self.ctc_wer_metric.write_stats(w)
                    with open(self.hparams.test_wer_file.replace("wer_", "cer_ctc_"), "w", encoding="utf-8") as w:
                        self.ctc_cer_metric.write_stats(w)

    def on_fit_batch_end(self, batch, outputs, loss, should_step):
        if should_step:
            self.hparams.noam_annealing(self.optimizer)


def dataio_prepare(hparams, tokenizer):
    data_folder = hparams["data_folder"]
    load = lambda p: sb.dataio.dataset.DynamicItemDataset.from_csv(csv_path=p, replacements={"data_root": data_folder})
    train_data = load(hparams["train_csv"])
    valid_data = load(hparams["valid_csv"]).filtered_sorted(sort_key="duration")
    test_datasets = {Path(p).stem: load(p).filtered_sorted(sort_key="duration") for p in hparams["test_csv"]}
    datasets = [train_data, valid_data] + list(test_datasets.values())
    valtest = [valid_data] + list(test_datasets.values())

    def read(wav):
        x, sr = sf.read(wav, dtype="float32", always_2d=False)
        assert sr == hparams["sample_rate"], f"{wav}: {sr} Hz"
        return torch.from_numpy(x)

    @sb.utils.data_pipeline.takes("wav")
    @sb.utils.data_pipeline.provides("sig")
    def audio_pipeline(wav):
        return read(wav)
    sb.dataio.dataset.add_dynamic_item(valtest, audio_pipeline)

    @sb.utils.data_pipeline.takes("wav")
    @sb.utils.data_pipeline.provides("sig")
    def audio_pipeline_train(wav):
        sig = read(wav)
        if "speed_perturb" in hparams:
            sig = hparams["speed_perturb"](sig.unsqueeze(0)).squeeze(0)
        return sig
    sb.dataio.dataset.add_dynamic_item([train_data], audio_pipeline_train)

    @sb.utils.data_pipeline.takes("wrd")
    @sb.utils.data_pipeline.provides("wrd", "tokens_list", "tokens_bos", "tokens_eos", "tokens")
    def text_pipeline(wrd):
        yield wrd
        tokens_list = tokenizer.encode_as_ids(wrd)
        yield tokens_list
        yield torch.LongTensor([hparams["bos_index"]] + tokens_list)
        yield torch.LongTensor(tokens_list + [hparams["eos_index"]])
        yield torch.LongTensor(tokens_list)
    sb.dataio.dataset.add_dynamic_item(datasets, text_pipeline)
    sb.dataio.dataset.set_output_keys(datasets, ["id", "sig", "wrd", "tokens_bos", "tokens_eos", "tokens"])

    train_bs = valid_bs = None
    if hparams["dynamic_batching"]:
        from speechbrain.dataio.sampler import DynamicBatchSampler
        train_bs = DynamicBatchSampler(train_data, length_func=lambda x: x["duration"], **hparams["dynamic_batch_sampler_train"])
        valid_bs = DynamicBatchSampler(valid_data, length_func=lambda x: x["duration"], **hparams["dynamic_batch_sampler_valid"])
    return train_data, valid_data, test_datasets, train_bs, valid_bs


if __name__ == "__main__":
    hparams_file, run_opts, overrides = sb.parse_arguments(sys.argv[1:])
    with open(hparams_file, encoding="utf-8") as fin:
        hparams = load_hyperpyyaml(fin, overrides)
    sb.utils.distributed.ddp_init_group(run_opts)
    sb.create_experiment_directory(experiment_directory=hparams["output_folder"],
                                   hyperparams_to_save=hparams_file, overrides=overrides)
    tokenizer = hparams["tokenizer"]
    tokenizer.load(hparams["bpe_model"])
    assert tokenizer.get_piece_size() == hparams["output_neurons"], \
        f"tokenizer has {tokenizer.get_piece_size()} pieces, output_neurons = {hparams['output_neurons']}"
    assert (tokenizer.unk_id(), tokenizer.bos_id(), tokenizer.eos_id()) == (0, hparams["bos_index"], hparams["eos_index"])

    train_data, valid_data, test_datasets, train_bs, valid_bs = dataio_prepare(hparams, tokenizer)
    hparams["test_search"] = hparams["valid_search"]          # no external LM (Zevallos et al. 2025)
    hparams["test_search"].beam_size = hparams["test_beam_size"]

    asr_brain = ASR(modules=hparams["modules"], opt_class=hparams["Adam"], hparams=hparams,
                    run_opts=run_opts, checkpointer=hparams["checkpointer"])
    asr_brain.tokenizer = tokenizer
    train_opts = hparams["train_dataloader_opts"]
    valid_opts = hparams["valid_dataloader_opts"]
    if train_bs is not None:
        train_opts = {"batch_sampler": train_bs, "num_workers": hparams["num_workers"]}
    if valid_bs is not None:
        valid_opts = {"batch_sampler": valid_bs}

    if not hparams["skip_train"]:
        asr_brain.fit(asr_brain.hparams.epoch_counter, train_data, valid_data,
                      train_loader_kwargs=train_opts, valid_loader_kwargs=valid_opts)

    os.makedirs(hparams["output_wer_folder"], exist_ok=True)
    for name, data in test_datasets.items():
        asr_brain.hparams.test_name = name
        asr_brain.hparams.test_wer_file = os.path.join(hparams["output_wer_folder"], f"wer_{name}.txt")
        test_opts = dict(hparams["test_dataloader_opts"])
        if "long" in name:
            test_opts["batch_size"] = 1
        asr_brain.evaluate(data, max_key="ACC", test_loader_kwargs=test_opts)



Writing /kaggle/working/Mamba-ASR/train_egy.py


In [10]:
# ---------------------------------------------------------------- 7. hparams and csv files for this run
OUT = f"{WORK}/results/{MODEL}"
if MODE == "smoke":
    OUT = f"{WORK}/smoke/{MODEL}"
    shutil.rmtree(OUT, ignore_errors=True)
    CSV_DIR = f"{WORK}/smoke/csv"; os.makedirs(CSV_DIR, exist_ok=True)
    sizes = {"train": 640, "dev": 160, "test": 160, "test_long": 16}
    for name, n in sizes.items():
        CSV[name].sample(n=n, random_state=0).to_csv(f"{CSV_DIR}/{name}.csv", index=False)
    run_epochs, warmup, vsi = 2, 100, 1
else:
    CSV_DIR = DATA
    run_epochs, warmup, vsi = EPOCHS, 30000, 10
    if RESUME_FROM:
        prev = sorted(glob.glob(f"/kaggle/input/{RESUME_FROM}/**/results/{MODEL}", recursive=True))
        assert prev, f"no results/{MODEL} under /kaggle/input/{RESUME_FROM}"
        if not os.path.isdir(OUT):
            shutil.copytree(prev[0], OUT)
        print("resuming from", prev[0], "| checkpoints:", sorted(os.listdir(f"{OUT}/save")) if os.path.isdir(f"{OUT}/save") else None)
    elif os.path.isdir(f"{OUT}/save") and os.listdir(f"{OUT}/save"):
        print("Found existing checkpoints in", f"{OUT}/save", ":", sorted(os.listdir(f"{OUT}/save")))

# Handle training data selection (Approach 3)
train_bs = 32
grad_accum = 1
if TRAIN_DATA == "mixed":
    MIXED_DIR = f"{WORK}/mixed_csv" if MODE == "full" else f"{WORK}/smoke/mixed_csv"
    os.makedirs(MIXED_DIR, exist_ok=True)
    for split, long_split in [("train", "train_long"), ("dev", "dev_long")]:
        df_s = CSV[split] if MODE == "full" else pd.read_csv(f"{CSV_DIR}/{split}.csv")
        df_l = CSV[long_split] if MODE == "full" else pd.read_csv(f"{CSV_DIR}/{long_split}.csv")
        pd.concat([df_s, df_l], ignore_index=True).to_csv(f"{MIXED_DIR}/{split}.csv", index=False)
    for test_s in ["test", "test_long"]:
        df_t = CSV[test_s] if MODE == "full" else pd.read_csv(f"{CSV_DIR}/{test_s}.csv")
        df_t.to_csv(f"{MIXED_DIR}/{test_s}.csv", index=False)
    CSV_DIR = MIXED_DIR
    train_bs, grad_accum = 4, 8  # 4 * 8 = 32 global batch, prevents OOM on 60s clips
    print(f"Using MIXED training data (train + train_long): batch_size={train_bs}, grad_accum={grad_accum}")
elif TRAIN_DATA == "train_long":
    LONG_DIR = f"{WORK}/long_csv" if MODE == "full" else f"{WORK}/smoke/long_csv"
    os.makedirs(LONG_DIR, exist_ok=True)
    (CSV["train_long"] if MODE == "full" else pd.read_csv(f"{CSV_DIR}/train_long.csv")).to_csv(f"{LONG_DIR}/train.csv", index=False)
    (CSV["dev_long"] if MODE == "full" else pd.read_csv(f"{CSV_DIR}/dev_long.csv")).to_csv(f"{LONG_DIR}/dev.csv", index=False)
    for test_s in ["test", "test_long"]:
        df_t = CSV[test_s] if MODE == "full" else pd.read_csv(f"{CSV_DIR}/{test_s}.csv")
        df_t.to_csv(f"{LONG_DIR}/{test_s}.csv", index=False)
    CSV_DIR = LONG_DIR
    train_bs, grad_accum = 4, 8  # 4 * 8 = 32 global batch, prevents OOM on 60s clips
    print(f"Using TRAIN_LONG training data: batch_size={train_bs}, grad_accum={grad_accum}")

HP = f"{WORK}/{MODEL}_{MODE}.yaml"
ejust_tools.make_hparams(f"{REPO}/hparams/S2S/{MODEL}.yaml", HP, data_folder=DATA, csv_dir=CSV_DIR, bpe_model=BPE,
                         output_folder=OUT, epochs=run_epochs, warmup=warmup, valid_search_interval=vsi,
                         batch_size=train_bs, grad_accumulation_factor=grad_accum,
                         long_decode_mode=LONG_DECODE_MODE, skip_train=EVAL_ONLY)
print("hparams:", HP, "| output:", OUT, "| decode mode:", LONG_DECODE_MODE, "| skip_train:", EVAL_ONLY)


hparams: /kaggle/working/conmamba_small_full.yaml | output: /kaggle/working/results/conmamba_small | decode mode: chunked | skip_train: False


In [11]:
# ---------------------------------------------------------------- 8. train + test
cmd = [sys.executable, "train_egy.py", HP, "--precision", "fp16", "--seed", str(SEED)]
if MODE == "full":
    cmd += ["--ckpt_interval_minutes", "30"]
limit = 3600 if MODE == "smoke" else int(TIME_LIMIT_H * 3600)
print(" ".join(cmd), "| time limit", limit, "s")
t0 = time.time()
log = open(f"{WORK}/{MODEL}_{MODE}_stdout.txt", "w")
p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in p.stdout:
        log.write(line)
        if "it/s" not in line and "s/it" not in line:      # hide progress-bar lines
            print(line, end="")
        if time.time() - t0 > limit:
            print(f"\n*** time limit reached ({limit} s): stopping; checkpoints are in {OUT}/save ***")
            p.terminate(); break
finally:
    p.wait(); log.close()
print(f"exit code {p.returncode} | {(time.time()-t0)/60:.1f} min")

/usr/bin/python3 train_egy.py /kaggle/working/conmamba_small_full.yaml --precision fp16 --seed 7775 --ckpt_interval_minutes 30 | time limit 39600 s
/kaggle/working/Mamba-ASR/modules/mamba/selective_scan_interface.py:184: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/kaggle/working/Mamba-ASR/modules/mamba/selective_scan_interface.py:253: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
/kaggle/working/Mamba-ASR/modules/mamba/selective_scan_interface.py:321: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/kaggle/working/Mamba-ASR/modules/mamba/selective_scan_interface.py:394: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(a

In [12]:
# ---------------------------------------------------------------- 9. summary (send this output to the team)
import re, glob, os, json
import pandas as pd

print("=== train_log.txt (last lines)")
tl = f"{OUT}/train_log.txt"
print(open(tl).read()[-3000:] if os.path.exists(tl) else "(none)")
for f in sorted(glob.glob(f"{OUT}/**/wer_*.txt", recursive=True)) + sorted(glob.glob(f"{OUT}/**/cer_*.txt", recursive=True)):
    print("===", os.path.basename(f)); print("".join(open(f, encoding="utf-8").readlines()[:3]))

results_summary = []
for f in sorted(glob.glob(f"{OUT}/**/wer_*.txt", recursive=True)):
    bname = os.path.basename(f)
    try:
        first_line = open(f, encoding="utf-8").readline()
        m = re.search(r"%WER\s+([\d\.]+)\s+\[\s*(\d+)\s*/\s*(\d+),\s*(\d+)\s*ins,\s*(\d+)\s*del,\s*(\d+)\s*sub\s*\]", first_line)
        if m:
            wer, errs, total, ins, dels, subs = m.groups()
            cer_file = f.replace("wer_", "cer_")
            cer_val = "-"
            if os.path.exists(cer_file):
                c_line = open(cer_file, encoding="utf-8").readline()
                cm = re.search(r"%WER\s+([\d\.]+)", c_line)
                if cm:
                    cer_val = cm.group(1)
            results_summary.append({
                "File": bname,
                "WER (%)": float(wer),
                "CER (%)": float(cer_val) if cer_val != "-" else "-",
                "Insertions": int(ins),
                "Deletions": int(dels),
                "Substitutions": int(subs),
                "Total Words": int(total),
            })
    except Exception as e:
        print(f"Error parsing {bname}: {e}")

if results_summary:
    print()
    print("=== BENCHMARK COMPARISON TABLE ===")
    print(pd.DataFrame(results_summary).to_string(index=False))

eff = [json.loads(l) for l in open(f"{OUT}/efficiency.jsonl")] if os.path.exists(f"{OUT}/efficiency.jsonl") else []
print()
print("=== EFFICIENCY SUMMARY ===")
print(pd.DataFrame(eff).to_string(index=False) if eff else "(no efficiency log)")
if MODE == "smoke" and eff:
    tr = [e for e in eff if e["stage"] == "TRAIN"]
    va = [e for e in eff if e["stage"] == "VALID"]
    sec_train = tr[-1]["seconds"] * len(CSV["train"]) / 640
    sec_valid = va[-1]["seconds"] * len(CSV["dev"]) / 160 if va else 0
    per_epoch = sec_train + 0.3 * sec_valid
    print()
    print(f"ESTIMATE for {MODEL}: about {sec_train/60:.0f} min training per epoch on this GPU, "
          f"peak GPU memory {max(e['peak_gpu_mb'] or 0 for e in tr)} MB")
    print(f"  -> {EPOCHS} epochs = about {EPOCHS*per_epoch/3600:.0f} GPU-hours, i.e. about "
          f"{EPOCHS*per_epoch/3600/TIME_LIMIT_H:.1f} Kaggle runs of {TIME_LIMIT_H:g} h")


=== train_log.txt (last lines)
s: 68880 - train loss: 5.82 - valid loss: 8.77, valid ACC: 8.87e-01
epoch: 85, lr: 5.25e-04, steps: 69700 - train loss: 5.75 - valid loss: 8.81, valid ACC: 8.87e-01
epoch: 86, lr: 5.22e-04, steps: 70520 - train loss: 5.66 - valid loss: 8.83, valid ACC: 8.86e-01
epoch: 87, lr: 5.19e-04, steps: 71340 - train loss: 5.63 - valid loss: 8.88, valid ACC: 8.87e-01
epoch: 88, lr: 5.16e-04, steps: 72160 - train loss: 5.58 - valid loss: 8.90, valid ACC: 8.86e-01
epoch: 89, lr: 5.13e-04, steps: 72980 - train loss: 5.51 - valid loss: 8.83, valid ACC: 8.85e-01
epoch: 90, lr: 5.10e-04, steps: 73800 - train loss: 5.52 - valid loss: 8.78, valid ACC: 8.86e-01, valid WER: 18.58, valid CER: 7.67
epoch: 91, lr: 5.07e-04, steps: 74620 - train loss: 5.49 - valid loss: 8.88, valid ACC: 8.86e-01
epoch: 92, lr: 5.04e-04, steps: 75440 - train loss: 5.34 - valid loss: 8.98, valid ACC: 8.86e-01
epoch: 93, lr: 5.02e-04, steps: 76260 - train loss: 5.39 - valid loss: 8.90, valid ACC: 8.